# GDAL

Description:
- Work with geospatial data

Resources:
- [Gdal Docs Tutorials] https://gdal.org/tutorials/index.html
- [Spatial Thoughts Mastering Gdal] https://courses.spatialthoughts.com/gdal-tools.html#introduction
- [Python Gdal / OGR Cookbook] https://pcjericks.github.io/py-gdalogr-cookbook/index.html

## Environment variables

This notebook reads these environment variables. Set them before you start Jupyter:

- `S3_BUCKET`: name of an S3 bucket you can access. Example 3 opens `ganymede-demo/sample.geojson` from it, so upload any GeoJSON file there first. AWS credentials come from the usual AWS configuration (environment variables, `~/.aws`, or an instance role).

## Example 1

Get the list of Supported GDAL drivers

In [ ]:
from osgeo import ogr

def get_gdal_drivers():
    # Initialize GDAL - mustbe inside of function
    ogr.UseExceptions()

    # Get the number of available drivers
    num_drivers = ogr.GetDriverCount()

    # Create a list to store driver names
    driver_list = []

    # Iterate through each driver and retrieve its name
    for i in range(num_drivers):
        driver = ogr.GetDriver(i)
        driver_name = driver.GetName()
        driver_list.append(driver_name)

    return driver_list

gdal_drivers = get_gdal_drivers()
print("GDAL Drivers:")
for driver in gdal_drivers:
    print(driver)

## Example 2

Open a remote Sentinel satellite raster dataset from s3 and create a numpy array

In [ ]:
from osgeo import gdal

# Open the remote path
remote_path = '/vsis3/sentinel-cogs/sentinel-s2-l2a-cogs/10/C/DA/2023/10/S2B_10CDA_20231002_0_L2A/AOT.tif'
ds = gdal.Open(remote_path)

# Get the band
band = ds.GetRasterBand(1)

# Create a numpy array
xoff, yoff, xcount, ycount = (0, 0, 10, 10)
np_array = band.ReadAsArray(xoff, yoff, xcount, ycount)
np_array

## Example 3

Open a remote vector dataset from S3

In [ ]:
import os
from osgeo import ogr

ogr.UseExceptions()

# GDAL reads the object straight from S3 through its /vsis3/ virtual filesystem
source_path_remote = f"/vsis3/{os.environ['S3_BUCKET']}/ganymede-demo/sample.geojson"

# Open the data (0 = read-only)
ds = ogr.Open(source_path_remote, 0)

layer = ds.GetLayer(0)
print("Layer:", layer.GetName())
print("Features:", layer.GetFeatureCount())
layer_defn = layer.GetLayerDefn()
print("Fields:", [layer_defn.GetFieldDefn(i).GetName() for i in range(layer_defn.GetFieldCount())])